# PPG CNN/LSTM — Kaggle Training → FP32 Weights → Pure C

**Input:** 1000 mẫu @125Hz (8 giây), output BPM. Giữ kiến trúc `raj5517/ppg-heart-rate-estimator`. 

Dữ liệu gốc là **synthetic PPG**. Notebook tự chứa source, không tải repo lúc chạy. Bật GPU trong Settings nếu có. Kaggle cần có TensorFlow + NumPy và GCC/make. Đọc `README_VI.md` trong project để xem nguồn dữ liệu, định dạng trọng số và giới hạn phần cứng.

Mặc định train checkpoint mới. Muốn smoke test: N=64, epochs=1, batch=8, benchmark=10; checkpoint này chưa đủ huấn luyện.


In [1]:
# Giải nén source đã đóng gói. Không truy cập mạng.
import base64, io, zipfile, pathlib, sys, subprocess, importlib.util
ROOT = pathlib.Path('/kaggle/working/ppg_c_project')
ROOT.mkdir(parents=True, exist_ok=True)
PAYLOAD = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    for name in archive.namelist():
        target = (ROOT / name).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Invalid archive path')
    archive.extractall(ROOT)
for dep in ['tensorflow', 'numpy']:
    if importlib.util.find_spec(dep) is None:
        raise RuntimeError(f'Missing {dep}; select the standard Kaggle Python image')
subprocess.run(['cc', '--version'], check=True)
subprocess.run(['make', '-C', str(ROOT)], check=True)
print('Source:', ROOT)


cc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

make: Entering directory '/kaggle/working/ppg_c_project'
mkdir -p build
cc -O3 -std=c11 -Wall -Wextra -Werror -ffp-contract=off -Ic c/ppg.c c/ppg_io.c c/main.c -lm -o build/ppg_infer
cc -O3 -std=c11 -Wall -Wextra -Werror -ffp-contract=off -fPIC -shared -Ic c/ppg.c c/ppg_io.c -lm -o build/libppg.so
make: Leaving directory '/kaggle/working/ppg_c_project'
Source: /kaggle/working/ppg_c_project


## Cấu hình training

- `synthetic`: tự tạo dữ liệu mô phỏng. Chia theo simulation, không có người thật.
- `real_npz`: dùng NPZ của bạn, phải có `X`, `y_bpm`, `subject_id`, `fs_hz=125`. Chia độc lập theo subject.
- Luôn xuất best checkpoint theo validation MAE; dùng FP32. Không thay TensorFlow GPU wheel bằng tensorflow-cpu.


In [2]:
MODE = 'synthetic'
DATA_PATH = ''  # real_npz: '/kaggle/input/<dataset>/<file>.npz'
N_SYNTHETIC = 10000
EPOCHS = 80
BATCH = 64
SEED = 42
TOLERANCE_BPM = 0.05
BENCHMARK_ITERS = 300
RESULTS = ROOT / 'results'


## Huấn luyện, xuất trọng số, kiểm chứng và benchmark

Cùng một checkpoint cho Keras và C. So sánh toàn bộ test windows + stress cases. Sai khác vượt tolerance sẽ dừng pipeline. Benchmark dùng CPU, batch=1, 1 thread, warm-up 20, bao gồm preprocessing và dispatch ở cả hai đường thực thi. Không suy ra speedup MCU từ benchmark này.


In [3]:
cmd = [sys.executable, str(ROOT / 'train_export_verify.py'),
       '--mode', MODE, '--data-path', DATA_PATH, '--n', str(N_SYNTHETIC),
       '--epochs', str(EPOCHS), '--batch', str(BATCH), '--seed', str(SEED),
       '--tolerance', str(TOLERANCE_BPM), '--benchmark-iters', str(BENCHMARK_ITERS),
       '--outdir', str(RESULTS)]
subprocess.run(cmd, cwd=ROOT, check=True)


I0000 00:00:1791808073.242683      63 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791808073.246481      63 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "PPG_HR_Estimator"


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/ppg_c_project/train_export_verify.py', '--mode', 'synthetic', '--data-path', '', '--n', '10000', '--epochs', '80', '--batch', '64', '--seed', '42', '--tolerance', '0.05', '--benchmark-iters', '300', '--outdir', '/kaggle/working/ppg_c_project/results'], returncode=0)

In [4]:
import json
num = json.loads((RESULTS / 'numerical_report.json').read_text())
bench = json.loads((RESULTS / 'benchmark_report.json').read_text())
print('Split:', num['split'])
print('Numerical PASS:', num['pass'])
print('Test windows:', num['test_windows'])
print('Keras MAE:', num['keras']['mae_bpm'])
print('C MAE:', num['pure_c']['mae_bpm'])
print('Max C-Keras difference:', num['max_abs_diff_bpm'])
print('Latency:', bench['latency'])
print('Static structure sizes:', bench['memory'])
print('First raw window, native C prediction:')
subprocess.run([str(ROOT / 'build/ppg_infer'), str(RESULTS / 'model_weights.bin'),
                str(RESULTS / 'first_test_window.csv')], check=True)


Split: {'seed': 42, 'fs_hz': 125, 'window_samples': 1000, 'provenance': {'kind': 'synthetic', 'generator': 'adapted upstream harmonic generator; exact arange/125 time axis', 'human_subjects': 0, 'split_unit': 'independent simulation', 'noise_fraction': 0.2034}, 'windows': {'test': 1500, 'val': 1500, 'train': 7000}}
Numerical PASS: True
Test windows: 1500
Keras MAE: 0.8303773371378581
C MAE: 0.8303757731119792
Max C-Keras difference: 3.0517578125e-05
Latency: {'keras': {'p50_ms': 5.0787505, 'p95_ms': 5.457444000000001, 'mean_ms': 5.11199058}, 'pure_c': {'p50_ms': 1.662722, 'p95_ms': 1.9516746, 'mean_ms': 1.6833703166666665}, 'p50_speedup': 3.054479642417674}
Static structure sizes: {'weight_payload_bytes': 101124, 'model_struct_bytes': 101124, 'workspace_struct_bytes': 69536, 'note': 'Static structures, NOT peak process RSS. MCU flash/linker usage must be measured separately.'}
First raw window, native C prediction:


CompletedProcess(args=['/kaggle/working/ppg_c_project/build/ppg_infer', '/kaggle/working/ppg_c_project/results/model_weights.bin', '/kaggle/working/ppg_c_project/results/first_test_window.csv'], returncode=0)

## Đóng gói kết quả

`model_weights.bin` dùng host C loader. `model_weights.h` dùng Flash-resident constants trong firmware. C không cần TensorFlow/TFLite tại runtime. Workspace ~67,91KiB; cần đo RAM, Flash, latency trên phần cứng đích. Đầu vào không đủ chất lượng phải bị gate bởi tầng sensor, không dùng output model cho input hằng để cảnh báo.


In [5]:
OUTPUT_ZIP = pathlib.Path('/kaggle/working/PPG_Kaggle_Result.zip')
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(ROOT.rglob('*')):
        if path.is_file() and '__pycache__' not in path.parts:
            z.write(path, pathlib.Path('ppg_c_project') / path.relative_to(ROOT))
print('Download from Kaggle Output:', OUTPUT_ZIP)
from IPython.display import FileLink, display
display(FileLink(str(OUTPUT_ZIP)))


Download from Kaggle Output: /kaggle/working/PPG_Kaggle_Result.zip


/kaggle/working/PPG_Kaggle_Result.zip

In [6]:
%cd ppg_c_project

/kaggle/working/ppg_c_project


In [7]:
!make clean
!make

rm -rf build
mkdir -p build
cc -O3 -std=c11 -Wall -Wextra -Werror -ffp-contract=off -Ic c/ppg.c c/ppg_io.c c/main.c -lm -o build/ppg_infer
cc -O3 -std=c11 -Wall -Wextra -Werror -ffp-contract=off -fPIC -shared -Ic c/ppg.c c/ppg_io.c -lm -o build/libppg.so


In [8]:
!./build/ppg_infer results/model_weights.bin results/first_test_window.csv

131.44783
